In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

```
입력:          28 × 28    ← MNIST 원본
 ↓ conv1 (padding=1, k=3, s=1)
              28 × 28    ← 변화 없음 (same padding)
 ↓ pool (2×2, stride=2)
              14 × 14    ← 반으로
 ↓ conv2 (padding=1, k=3, s=1)
              14 × 14    ← 변화 없음
 ↓ pool (2×2, stride=2)
               7 × 7    ← 또 반으로
 ↓ flatten
              16 × 7 × 7 = 784
```

In [26]:
class CNN(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=6, \
            kernel_size=3, stride=1, padding=1
        )
        self.pool = nn.MaxPool2d(kernel_size=(2,2), stride=(2,2))
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, \
            kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(16*7*7, num_classes)

        self.initialize_weights()
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.reshape(x.shape[0], -1)    # [B, C, W, H] -> [B, feature]
        x = self.fc1(x)                  # [B, 16, 7, 7]  →  [B, 16*7*7]  =  [B, 784]

        return x    
    
    def initialize_weights(self):
        for m in self.modules():
            print(m)
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

In [27]:
if __name__ == '__main__':
    model = CNN(in_channels=3, num_classes=10)

CNN(
  (conv1): Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=784, out_features=10, bias=True)
)
Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
Conv2d(6, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
Linear(in_features=784, out_features=10, bias=True)


왜 weight 초기화가 중요한가
신경망 학습 시작할 때 weight 를 어떻게 초기화하느냐가 학습 성패를 좌우 할 수 있어요.

전부 0 으로 초기화하면? — 모든 뉴런이 똑같이 움직여서 학습이 안 됨 (symmetry 문제).

너무 큰 값으로 초기화하면? — activation 값이 점점 커져서 gradient explosion.

너무 작은 값으로 초기화하면? — activation 이 점점 작아져서 gradient vanishing. 깊은 네트워크에서 앞쪽 레이어가 학습 안 됨.

그래서 "적절한 분산을 갖는 랜덤값" 으로 초기화해야 하는데, 이 "적절한" 이 뭐냐가 초기화 방법의 핵심이에요.

Kaiming 초기화의 아이디어
핵심 아이디어: "매 레이어를 통과한 후에도 activation 의 분산이 일정하게 유지되도록" weight 분산을 정해주자.

이러면 깊은 네트워크에서도 신호가 약해지거나 폭발하지 않고 잘 전달돼요.

특히 Kaiming 은 ReLU 에 최적화 돼있어요. ReLU 가 음수를 전부 0으로 날려버리니까, 분산이 반토막 나요. 이걸 보상하려고 weight 분산을 2배로 늘려서 초기화하는 거예요.